# Inspect a generated iBeamLab dataset

This notebook opens the newest completed dataset created by `generate.py` and plots RBS and NRA spectra from several selected samples. Shorter raw spectra are zero-padded by `DatasetReader` to the maximum length observed for each method.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd().resolve()
if ROOT.name == "examples":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from ibeamlab.datasets import DatasetReader

## Select and load a dataset

Set `DATASET_PATH` to a directory if you do not want to use the newest completed dataset.

In [ ]:
DATASET_PATH: Path | None = None

if DATASET_PATH is None:
    dataset_root = ROOT / "examples" / "datasets"
    candidates = sorted(
        (path for path in dataset_root.iterdir() if path.is_dir()),
        key=lambda path: path.stat().st_mtime,
        reverse=True,
    )
    for candidate in candidates:
        try:
            candidate_reader = DatasetReader(candidate)
        except (RuntimeError, OSError):
            continue
        if candidate_reader.metadata.complete:
            DATASET_PATH = candidate
            break

if DATASET_PATH is None:
    raise FileNotFoundError(
        "No completed dataset found. Run `python examples/generate.py` first."
    )

reader = DatasetReader(DATASET_PATH)
metadata = reader.metadata
records = sorted(reader.read_all(), key=lambda record: record.sample_index)

print(f"Dataset: {DATASET_PATH}")
print(f"Samples: {len(records)} accepted / {metadata.requested} requested")
print(f"Invalid: {metadata.invalid}; failed: {metadata.failed}")
print(f"Methods: {list(metadata.spectrum_labels)}")
print(f"Padded spectrum lengths: {list(metadata.spectrum_lengths)}")
print(f"Open parameters: {list(metadata.parameter_names)}")

## Select representative samples

In [ ]:
NUMBER_OF_SAMPLES = 6

if not records:
    raise RuntimeError("The dataset contains no accepted samples.")

positions = np.unique(
    np.linspace(0, len(records) - 1, min(NUMBER_OF_SAMPLES, len(records)), dtype=int)
)
selected = [records[int(position)] for position in positions]

for record in selected:
    parameters = dict(zip(metadata.parameter_names, record.open_parameters, strict=True))
    print(record.sample_id, parameters)

## Plot RBS and NRA spectra

Every panel uses channel number on the horizontal axis and simulated counts on the vertical axis.

In [ ]:
method_count = len(metadata.spectrum_labels)
fig, axes = plt.subplots(
    method_count,
    1,
    figsize=(13, 4.5 * method_count),
    squeeze=False,
    constrained_layout=True,
)

for method_index, method_label in enumerate(metadata.spectrum_labels):
    axis = axes[method_index, 0]
    for record in selected:
        spectrum = record.result.spectra[method_index]
        counts = np.asarray(spectrum.counts, dtype=np.float32)
        axis.plot(np.arange(counts.size), counts, linewidth=1.0, label=record.sample_id)
    axis.set_title(f"{method_label} spectra")
    axis.set_xlabel("Channel")
    axis.set_ylabel("Counts")
    axis.grid(alpha=0.25)
    axis.legend(ncols=2, fontsize=8)

plt.show()

## Plot one selected sample separately

Change `SELECTED_INDEX` to inspect another accepted sample.

In [ ]:
SELECTED_INDEX = 0
record = records[SELECTED_INDEX]

fig, axes = plt.subplots(
    len(record.result.spectra),
    1,
    figsize=(13, 4.5 * len(record.result.spectra)),
    squeeze=False,
    constrained_layout=True,
)
for index, spectrum in enumerate(record.result.spectra):
    counts = np.asarray(spectrum.counts, dtype=np.float32)
    axes[index, 0].plot(counts, color=f"C{index}", linewidth=1.0)
    axes[index, 0].set(
        title=f"{record.sample_id} — {spectrum.label}",
        xlabel="Channel",
        ylabel="Counts",
    )
    axes[index, 0].grid(alpha=0.25)
plt.show()

In [ ]:
''